In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
import os
from IPython.display import Markdown, display
from datetime import datetime
load_dotenv(override=True)


In [ ]:
params = {"command": "uvx","args": ["code-index-mcp"]}

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

mcp_tools

In [ ]:
instructions = "Set the project path to this: /home/misi/src/learning/udemy/llm_ed_donner_agents/6_mcp/community_contributions"
request = "Find all Python files which has potential security problems in the project and list their paths."
model = "gpt-4.1-mini"

In [ ]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

In [ ]:
import gradio as gr

async def chat(message, history):
    conversation = [
        {"role": item["role"], "content": item["content"]}
        for item in history
    ] + [{"role": "user", "content": message}]
    async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
        agent = Agent(
            name="code_index_chat",
            instructions=instructions,
            model=model,
            mcp_servers=[mcp_server],
        )
        with trace("code_index_chat"):
            result = await Runner.run(agent, conversation)
    return result.final_output

gr.ChatInterface(
    fn=chat,
    type="messages",
    title="Code Index MCP Chat",
    description="Ask questions about the indexed project. The agent uses code-index-mcp tools when needed.",
).launch()